# cytozip Python API

### view header

In [1]:
import os,sys
os.chdir(os.path.expanduser("~/Projects/Github/cytozip/cytozip_example_data"))
import cytozip as czip
reader=czip.Reader("output/cz/FC_P13a_3C_7-1-A11-O1.cz")
reader.print_header()

magic  :  b'CZIP'
version  :  0.3
total_size  :  31467075
message  :  mm10_with_chrL.allc.cz
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom']
header_size  :  61


In [2]:
reader.header

{'magic': b'CZIP',
 'version': 0.3,
 'total_size': 31467075,
 'message': 'mm10_with_chrL.allc.cz',
 'formats': ['B', 'B'],
 'columns': ['mc', 'cov'],
 'sort_col': None,
 'delta_cols': [],
 'chunk_dims': ['chrom'],
 'header_size': 61}

### summary chunks & blocks

In [3]:
df=reader.summary_chunks(printout=False)
df

,chrom,chunk_start_offset,chunk_size,chunk_tail_offset,chunk_nblocks,chunk_nrows
chunk_dims,,,,,,
"(chr1,)",chr1,61,2093475,2098381,603,78962721
"(chr10,)",chr10,2098381,1539301,3640920,402,52609184
"(chr11,)",chr11,3640920,1804058,5448176,397,52027265
"(chr12,)",chr12,5448176,1402168,6853350,373,48799752
"(chr13,)",chr13,6853350,1549591,8405939,372,48750883
...,...,...,...,...,...,...
"(chrUn_GL456379,)",chrUn_GL456379,31426215,464,31426718,1,26755
"(chrUn_GL456366,)",chrUn_GL456366,31426718,410,31427167,1,16990
"(chrUn_GL456368,)",chrUn_GL456368,31427167,400,31427606,1,7478


Every chunk has a dimension (chrom, sample, cell types or the combination of those dimensions)

In [4]:
df=reader.summary_blocks(printout=False)
df

,chunk_dims,block_start_offset,block_size,block_data_start,block_data_len
0,"(chr1,)",71,4221,0,262143
1,"(chr1,)",4292,3645,262143,262143
2,"(chr1,)",7937,3416,524286,262143
3,"(chr1,)",11353,4007,786429,262143
4,"(chr1,)",15360,4307,1048572,262143
...,...,...,...,...,...
8469,"(chrUn_GL456379,)",31426225,454,0,53510
8470,"(chrUn_GL456366,)",31426728,400,0,33980
8471,"(chrUn_GL456368,)",31427177,390,0,14956
8472,"(chrUn_JH584304,)",31427616,26576,0,96524


### query

In [5]:
?reader.query

Signature:
reader.query(
    chunk_key=None,
    start=None,
    end=None,
    regions=None,
    query_col=[0],
    reference=None,
    printout=True,
)
Docstring:
query .cz file by chunk_key, start and end, if regions provided, chunk_key, start and
end should be None, regions should be a list, each element of regions
is a list, for example, regions=[[('cell1','chr1'),1,10],
[('cell10','chr22'),100,200]],and so on.

Coordinate semantics
--------------------
``start`` / ``end`` are matched **inclusively** on the values
stored in the column(s) given by ``query_col`` — i.e.
``[start, end]``, **not** the BED-style 0-based half-open
``[start, end)``. The coordinate base (0-based vs 1-based)
is whatever was stored when the ``.cz`` file was built:

* ALLC-derived ``.cz`` (``allc_to_cz`` / ``bam_to_cz`` /
  reference C-position file) is **1-based**.
* BED-derived ``.cz`` preserves the source coordinates as-is
  (typically 0-based), but cytozip still applies the bounds
  *inclusively*.

This di

In [6]:
for record in reader.query(chunk_key="chr9",start=3000294,end=3000472,
                           reference="output/mm10_with_chrL.allc.cz",
                           printout=False):
    print(record) # list

['chr9', '3000294', '-', 'CAT', '13', '22']
['chr9', '3000296', '+', 'CCT', '30', '40']
['chr9', '3000297', '+', 'CTA', '29', '40']
['chr9', '3000300', '+', 'CAA', '30', '40']
['chr9', '3000304', '-', 'CAT', '4', '30']
['chr9', '3000305', '-', 'CCA', '4', '30']
['chr9', '3000307', '+', 'CAT', '33', '41']
['chr9', '3000312', '+', 'CTA', '36', '45']
['chr9', '3000321', '+', 'CCA', '41', '49']
['chr9', '3000322', '+', 'CAA', '42', '49']
['chr9', '3000325', '+', 'CTT', '43', '51']
['chr9', '3000331', '+', 'CAG', '44', '51']
['chr9', '3000333', '-', 'CTG', '3', '33']
['chr9', '3000338', '+', 'CCT', '50', '57']
['chr9', '3000339', '+', 'CTC', '49', '57']
['chr9', '3000341', '+', 'CGC', '51', '53']
['chr9', '3000342', '-', 'CGA', '29', '31']
['chr9', '3000343', '+', 'CCA', '47', '58']
['chr9', '3000344', '+', 'CAT', '50', '60']
['chr9', '3000351', '+', 'CAC', '49', '61']
['chr9', '3000353', '+', 'CGT', '56', '62']
['chr9', '3000354', '-', 'CGT', '29', '31']
['chr9', '3000356', '+', 'CCT', '52

## Numpy / DataFrame APIs

The numpy-level APIs let you decode whole chunks (or a single interval) into zero-copy numpy arrays without going through Python record tuples. They are the fastest way to pull data out of a `.cz` file for downstream pandas / scikit-learn / scanpy work.


### `chunk2numpy(dims)` — decode a whole chunk

Returns a 1-D numpy *structured* array named positionally (`f0`, `f1`, ...) and align with `reader.header["columns"]`. The buffer is a zero-copy view onto the decompressed chunk bytes — use `chunk2df(...)` for pandas column names.

In [7]:
?reader.chunk2numpy

Signature: reader.chunk2numpy(dims, reformat=False)
Docstring:
Read an entire chunk as a numpy structured ndarray.

This is the fastest single-cell read path: it skips the per-row
Python tuple construction and the ``pd.DataFrame`` build that
dominate :meth:`chunk2df`. Internally it calls
:meth:`fetch_chunk_bytes` (which already runs DELTA decode and the
Cython block fetcher) and views the raw bytes as a structured
dtype with one field per column.

Parameters
----------
dims : tuple
        Chunk key (e.g. ``('chr1',)``).
reformat : bool, default False
        If True, decode bytes-typed columns ('s'/'c') into Python
        str (returned as a separate ``object`` ndarray, since numpy
        structured arrays can't hold variable-length str directly).
        Most analysis pipelines should leave this False and operate
        on the raw bytes columns.

Returns
-------
np.ndarray
        A structured ndarray of length ``len(chunk_bytes) //
        unit_size``. Field names are ``f0, f1, ..

In [8]:
arr = reader.chunk2numpy(dims=("chr1",))
print("dtype:", arr.dtype)
print("n records:", arr.shape[0])
print("first 5 mc:", arr["f0"][:5])
print("first 5 cov:", arr["f1"][:5])
arr

dtype: [('f0', 'u1'), ('f1', 'u1')]
n records: 78962721
first 5 mc: [0 0 0 0 0]
first 5 cov: [0 0 0 0 0]


array([(0, 0), (0, 0), (0, 0), ..., (0, 0), (0, 0), (0, 0)],
      dtype=[('f0', 'u1'), ('f1', 'u1')])

In [9]:
reader.chunk2numpy(dims=("chr1",),reformat=True)

{'mc': array([0, 0, 0, ..., 0, 0, 0], dtype=uint8),
 'cov': array([0, 0, 0, ..., 0, 0, 0], dtype=uint8)}

In [11]:
reader2=czip.Reader("output/all_cells.cz")
reader2.chunk_info

,chrom,cell_id,chunk_start_offset,chunk_size,chunk_tail_offset,chunk_nblocks,chunk_nrows
chunk_dims,,,,,,,
"(chr1, FC_E17b_3C_5-5-I24-A21)",chr1,FC_E17b_3C_5-5-I24-A21,47,6002191,6007106,603,78962721
"(chr10, FC_E17b_3C_5-5-I24-A21)",chr10,FC_E17b_3C_5-5-I24-A21,6007106,3999949,10010316,402,52609184
"(chr11, FC_E17b_3C_5-5-I24-A21)",chr11,FC_E17b_3C_5-5-I24-A21,10010316,3835747,13849284,397,52027265
"(chr12, FC_E17b_3C_5-5-I24-A21)",chr12,FC_E17b_3C_5-5-I24-A21,13849284,3618595,17470908,373,48799752
"(chr13, FC_E17b_3C_5-5-I24-A21)",chr13,FC_E17b_3C_5-5-I24-A21,17470908,3703795,21177724,372,48750883
...,...,...,...,...,...,...,...
"(chrUn_GL456379, FC_P28a_3C_2-1-E5-N14)",chrUn_GL456379,FC_P28a_3C_2-1-E5-N14,277403513,357,277403931,1,26755
"(chrUn_GL456366, FC_P28a_3C_2-1-E5-N14)",chrUn_GL456366,FC_P28a_3C_2-1-E5-N14,277403931,476,277404468,1,16990
"(chrUn_GL456368, FC_P28a_3C_2-1-E5-N14)",chrUn_GL456368,FC_P28a_3C_2-1-E5-N14,277404468,265,277404794,1,7478


In [12]:
# convert a given chunk (a combination of chromosome and cell_id) into a numpy array
reader2.chunk2numpy(dims=("chr9",'FC_E17b_3C_5-5-I24-A21'),reformat=True)

{'mc': array([1, 4, 4, ..., 0, 0, 0], dtype=uint8),
 'cov': array([1, 4, 5, ..., 0, 0, 0], dtype=uint8)}

### `chunk2df(dims)` — decode as a DataFrame

Thin wrapper around `chunk2numpy`. Each numeric field is wrapped in a pandas Series with **no copy**; setting `reformat=True` decodes any byte-string columns to UTF-8 (slower; only needed when the chunk has `S<n>` columns such as `strand` / `context`).

In [13]:
df = reader.chunk2df(dims=("chr1",))
df.head()

,mc,cov
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [16]:
df = reader2.chunk2df(dims=("chr9",'FC_E17b_3C_5-5-I24-A21'))
df.head()

,mc,cov
0,1,1
1,4,4
2,4,5
3,8,8
4,6,11


### `query_numpy(chunk_key, start, end, reference=...)`

Range-query a single chunk by position and return
``(positions, records)`` as numpy arrays.

* ``start`` / ``end`` are **inclusive**, **1-based** for ALLC-derived `.cz` (`allc_to_cz` / `bam_to_cz`).
* Position lookup uses `np.searchsorted` on the cached sort column, so the cost is `O(log n)` plus the size of the slice.


In [23]:
?reader.query_numpy

Signature: reader.query_numpy(chunk_key, start, end, reference=None, sort_col=None)
Docstring:
Vectorized region query — returns a structured ``np.ndarray``.

~10× faster than :meth:`query` for warm in-process queries
because it bypasses the per-record Python tuple loop and uses
``np.searchsorted`` on a cached whole-chunk numpy array.

Coordinate semantics
--------------------
``start`` / ``end`` are matched **inclusively** against the
values stored in the sort column — i.e. ``[start, end]``,
**not** the BED-style 0-based half-open ``[start, end)``.
The coordinate base (0-based vs 1-based) is whatever was
stored when the ``.cz`` file was built:

* ``.cz`` produced by ``allc_to_cz`` / ``bam_to_cz`` /
  the AllC reference builder uses **1-based** positions
  (matching the ALLCools / tabix-on-allc.tsv.gz convention).
* ``.cz`` produced from BED-like inputs preserves the source
  coordinates as-is (typically 0-based), but bounds are still
  applied *inclusively*.

This differs from ``pytab

In [20]:
pos, recs = reader.query_numpy(
    "chr1", 3000294, 3010000, reference="output/mm10_with_chrL.allc.cz") # query a regions
print("n hits:", pos.size)
print("pos[:5]:", pos[:5])
print("recs[:5]:", recs[:5])

n hits: 3332
pos[:5]: [3000296 3000299 3000305 3000310 3000311]
recs[:5]: [(0, 0) (0, 0) (0, 0) (0, 0) (0, 0)]


In [21]:
pos

array([3000296, 3000299, 3000305, ..., 3009984, 3009992, 3009993],
      dtype=uint64)

In [22]:
recs # mc and cov

array([(0, 0), (0, 0), (0, 0), ..., (0, 0), (0, 0), (0, 0)],
      dtype=[('f0', 'u1'), ('f1', 'u1')])

### `query_numpy_chunk_batch(chunk_key, intervals, reference=...)`

Batched version: runs many `[start, end]` queries against the **same** chunk in one shot. The chunk's sort column is decoded (or pulled from cache) only once, so `N` intervals cost roughly `O(decode + N log n)` instead of `N · O(decode + log n)`.

In [24]:
intervals = [(3_000_000, 3_010_000),
             (4_000_000, 4_005_000),
             (5_000_000, 5_001_000)]
results = reader.query_numpy_chunk_batch(
    "chr1", intervals, reference="output/mm10_with_chrL.allc.cz")
for (s, e), (pos, recs) in zip(intervals, results):
    print(f"chr1:{s}-{e}: {pos.size} hits")

chr1:3000000-3010000: 3440 hits
chr1:4000000-4005000: 1942 hits
chr1:5000000-5001000: 405 hits


In [25]:
results

[(array([3000003, 3000005, 3000009, ..., 3009984, 3009992, 3009993],
        dtype=uint64),
  array([(0, 0), (0, 0), (0, 0), ..., (0, 0), (0, 0), (0, 0)],
        dtype=[('f0', 'u1'), ('f1', 'u1')])),
 (array([4000005, 4000006, 4000009, ..., 4004996, 4004998, 4005000],
        dtype=uint64),
  array([(0, 0), (0, 1), (0, 1), ..., (0, 0), (0, 0), (0, 0)],
        dtype=[('f0', 'u1'), ('f1', 'u1')])),
 (array([5000004, 5000007, 5000009, 5000019, 5000026, 5000027, 5000029,
         5000033, 5000040, 5000041, 5000042, 5000045, 5000052, 5000054,
         5000055, 5000056, 5000057, 5000059, 5000061, 5000062, 5000064,
         5000066, 5000067, 5000070, 5000071, 5000073, 5000074, 5000076,
         5000077, 5000080, 5000081, 5000082, 5000089, 5000090, 5000091,
         5000093, 5000094, 5000096, 5000099, 5000100, 5000101, 5000103,
         5000104, 5000105, 5000107, 5000109, 5000110, 5000111, 5000113,
         5000114, 5000117, 5000122, 5000124, 5000126, 5000131, 5000134,
         5000136, 5000

### Decoding many chunks in parallel — `chunk2numpy` + `ThreadPoolExecutor`

`chunk2numpy` releases the GIL during zlib / mmap I/O, so a
`ThreadPoolExecutor` is enough to scan many chunks in parallel without spawning subprocesses.  This pattern replaces the old `chunks2numpy` helper:


In [26]:
from concurrent.futures import ThreadPoolExecutor
import glob

def _decode(path, dim):
    """Worker: each thread opens its own Reader (cheap)."""
    with czip.Reader(path) as r:
        arr = r.chunk2numpy(dim)
        return path, dim, arr["f0"].sum(), arr["f1"].sum()

paths = sorted(glob.glob("output/cz/FC_*.cz"))
dim = ("chr1",)

with ThreadPoolExecutor(max_workers=4) as ex:
    futs = [ex.submit(_decode, p, dim) for p in paths]
    rows = [f.result() for f in futs]

import pandas as pd
pd.DataFrame(rows, columns=["cz", "dim", "mc_sum", "cov_sum"])

,cz,dim,mc_sum,cov_sum
0,output/cz/FC_E17b_3C_5-5-I24-A21.cz,"(chr1,)",7587763,9531202
1,output/cz/FC_M_E15a_3C_1-1-I5-B1.cz,"(chr1,)",972175,1942714
2,output/cz/FC_M_P12b_3C_2-5-M17-N10.cz,"(chr1,)",1493004,2957171
3,output/cz/FC_M_P3b_3C_6-6-J3-P24.cz,"(chr1,)",870487,1743698
4,output/cz/FC_M_P6a_3C_7-3-K21-P5.cz,"(chr1,)",488349,983148
5,output/cz/FC_M_P9B_3C_6-2-F6-O4.cz,"(chr1,)",183837,359090
6,output/cz/FC_P0b_3C_5-1-I24-J14.cz,"(chr1,)",3098179,6198496
7,output/cz/FC_P13a_3C_7-1-A11-O1.cz,"(chr1,)",1361921,2733335
8,output/cz/FC_P28a_3C_2-1-E5-N14.cz,"(chr1,)",1189905,2349934


### `align_cz` — column-concatenate two reference-aligned `.cz` files

Two reference-*less* `.cz` files (per-cell `mc`/`cov` produced by `allc2cz` with a `reference=`) share the same reference axis — one row per reference position, in reference order. `align_cz` attaches the `pos` column decoded from the reference and writes the two files' data columns side-by-side. The write path is a vectorized byte-level `hstack` (no per-column struct decode / re-encode), so it runs close to memory bandwidth.

* Pass `output=` a path to write a new `.cz`; colliding column names (e.g. `mc`, `cov`) get the `suffixes` (default `_1` / `_2`).
* Pass `output=None` (default) to get a `pandas.DataFrame` back instead of writing a file.
* `pos_col` selects the reference's coordinate column (default `'pos'`; pass a name or 0-based index if your reference names it differently).

In [ ]:
?czip.align_cz

In [ ]:
# Write mode: join two per-cell .cz files into one, attaching pos from the reference
czip.align_cz(
    input1="output/cz/FC_P13a_3C_7-1-A11-O1.cz",
    input2="output/cz/FC_E17b_3C_5-5-I24-A21.cz",
    output="output/FC_P13a_vs_E17b.aligned.cz",
    reference="output/mm10_with_chrL.allc.cz",
)
out = czip.Reader("output/FC_P13a_vs_E17b.aligned.cz")
print("columns:", out.header["columns"])
out.chunk2df(dims=("chr1",)).head()

In [ ]:
# DataFrame mode: output=None returns a pandas DataFrame directly (no file written)
df = czip.align_cz(
    input1="output/cz/FC_P13a_3C_7-1-A11-O1.cz",
    input2="output/cz/FC_E17b_3C_5-5-I24-A21.cz",
    reference="output/mm10_with_chrL.allc.cz",
)
print("shape:", df.shape)
df.head()

### Pack .allc.tsv.gz to .cz without coordinates (using reference)

In [27]:
? czip.allc2cz

Signature:
 czip.allc2cz(
    input,
    output,
    reference=None,
    missing_value=[0, 0],
    formats=['B', 'B'],
    columns=['mc', 'cov'],
    chunk_dims=['chrom'],
    usecols=[4, 5],
    ref_pos_col=0,
    allc_pos_col=1,
    sep='\t',
    chrom_order=None,
    batch_size=5000,
    sort_col=None,
    delta_cols=None,
    jobs=1,
    pattern='*.allc.tsv.gz',
    skip_existing=True,
    _ref_pos_dict=None,
)
Docstring:
convert allc.tsv.gz to .cz file.

When ``input`` is a directory, ALL matching allc files in the directory
are converted in parallel. The reference .cz (if given) is decoded once
in the parent process and shared with workers via fork-based copy-on-write,
so the reference memory cost is paid only once regardless of ``jobs``.

Parameters
----------
input : path
    Path to allc.tsv.gz (must have .tbi index), OR a directory containing
    many allc.tsv.gz files (batch mode).
output : path
    Output .cz file (single-file mode), or output directory (batch mode).
refere

## Convert .cz into allc using Python API

In [28]:
reader=czip.Reader("output/cz/FC_P13a_3C_7-1-A11-O1.cz")

In [29]:
reader.header

{'magic': b'CZIP',
 'version': 0.3,
 'total_size': 31467075,
 'message': 'mm10_with_chrL.allc.cz',
 'formats': ['B', 'B'],
 'columns': ['mc', 'cov'],
 'sort_col': None,
 'delta_cols': [],
 'chunk_dims': ['chrom'],
 'header_size': 61}

In [30]:
?reader.to_bgzip

Signature:
reader.to_bgzip(
    output,
    reference=None,
    chunk_order=None,
    where=None,
    tabix=True,
    cov_col=None,
    allc_format=False,
)
Docstring:
Convert a .cz file to a bgzip-compressed TSV (``.tsv.gz``).

The output is bgzip (BGZF) compressed and tabix-indexable. The
exact column layout is data-driven, so this works for both
BS-seq (allc.tsv.gz) and methylation-array (probe-level beta)
.cz files. The output column order is::

  chrom  [ref_columns...]  [data_columns...]  [mc_flag if allc_format]

Common recipes:

* ALLCools allc.tsv.gz (7-column format, trailing ``mc_flag=1``)::

    reader.to_bgzip("x.allc.tsv.gz", reference=ref_cz,
                    cov_col="cov", allc_format=True)

* Methylation array beta TSV (no row filtering, no trailing col)::

    reader.to_bgzip("x.beta.tsv.gz", reference=ref_cz)

Performance
-----------
Instead of iterating record-by-record via :meth:`fetch`, this
method uses a vectorised pipeline:

1. :meth:`fetch_chunk_bytes` — bul

In [31]:
%time reader.to_bgzip(output="output/cz/FC_P13a_3C_7-1-A11-O1.allc.tsv.gz", \
                     reference="output/mm10_with_chrL.allc.cz",cov_col='cov')

CPU times: user 1min 27s, sys: 11.4 s, total: 1min 39s
Wall time: 1min 39s


Find the difference between cz-to-allc'd allc file and the original allc file
```shell
norm() { zcat "$1" | awk 'BEGIN{OFS="\t"}$6>0{print $1,$2,$3,$4,$5,$6}' | sort -k1,1 -k2,2n; }
diff <(norm output/allc/FC_P13a_3C_7-1-A11-O1.allc.tsv.gz) <(norm output/cz/FC_P13a_3C_7-1-A11-O1.allc.tsv.gz) | head
```
In .cz, the mc and cov columns are typically stored as uint8 (range 0–255) or uint16 (0–65 535) for compactness. The original allc.tsv.gz has no such limit (parsed as 64-bit integers).

Single-cell allc: per-site coverage rarely exceeds 255 → uint8 is safe, no difference.

Pseudobulk / merged allc: high-coverage sites can easily exceed uint8 (and sometimes uint16). Without choosing a wider dtype at write time (uint32), values may saturate or wrap around, causing mc / cov mismatches in the round-trip.

In [37]:
# For example, we query a regions showing different records between two allc files
! tabix output/allc/FC_P13a_3C_7-1-A11-O1.allc.tsv.gz chr12:3109881-3109893

chr12	3109881	+	CAC	245	331	1
chr12	3109883	+	CGG	381	399	1
chr12	3109884	-	CGT	225	253	1
chr12	3109885	-	CCG	99	262	1
chr12	3109891	-	CAT	92	309	1
chr12	3109893	-	CTC	90	318	1


In [38]:
! tabix output/cz/FC_P13a_3C_7-1-A11-O1.allc.tsv.gz chr12:3109881-3109893

chr12	3109881	+	CAC	245	255
chr12	3109883	+	CGG	255	255
chr12	3109884	-	CGT	225	253
chr12	3109885	-	CCG	99	255
chr12	3109891	-	CAT	92	255
chr12	3109893	-	CTC	90	255
